## Sentiment Analysis

### Install NLTK Resources

In [ ]:
import nltk
nltk.download('vader_lexicon')

### Compute Sentiment

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

df["sentiment"] = df["headline"].apply(
    lambda x: sia.polarity_scores(x)["compound"]
)

## Daily Sentiment Aggregation

In [ ]:
daily_sentiment = (
    df.groupby(["stock", df["date"].dt.date])["sentiment"]
    .mean()
    .reset_index()
)

## Daily Returns

In [ ]:
stock_df["daily_return"] = (
    stock_df["Adj Close"].pct_change() * 100
)

## Merge News + Price Data

In [ ]:
merged = pd.merge(
    daily_sentiment,
    stock_df,
    left_on="date",
    right_on="Date"
)

## Correlation

In [ ]:
correlation = merged["sentiment"].corr(
    merged["daily_return"]
)

print(correlation)

## Scatter Plot

In [ ]:
plt.scatter(
    merged["sentiment"],
    merged["daily_return"]
)

plt.xlabel("Sentiment")
plt.ylabel("Daily Return")
plt.title(f"Correlation: {correlation:.2f}")

plt.show()

## Handling Weekend News

In [ ]:
from pandas.tseries.offsets import BDay

df["aligned_date"] = df["date"]

df.loc[
    df["date"].dt.weekday >= 5,
    "aligned_date"
] = df["date"] + BDay(1)